# 02 Scratch QLoRA Experiments

Run a readable from-scratch QLoRA experiment with NF4 quantization, FP16 LoRA adapters, and a paged optimizer.

In [ ]:
from pathlib import Path
import sys
import torch

PROJECT_ROOT = Path("/content/qLoRA").resolve()
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")])


In [ ]:
from qlora_scratch.train import ExperimentConfig

config = ExperimentConfig(
    model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_seq_length=256,
    train_batch_size=2,
    eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    epochs=1,
    max_train_samples=512,
    max_eval_samples=128,
    lora_rank=8,
    lora_alpha=16,
    quant_block_size=64,
    optimizer_page_size=2**18,
    output_dir=str(PROJECT_ROOT / "results" / "scratch_qlora"),
)
config


In [ ]:
sample_prompts = [
    "### Instruction:\nSummarize QLoRA in plain language for a student.\n\n### Response:\n",
    "### Instruction:\nWhat is the purpose of NF4 quantization in QLoRA?\n\n### Response:\n",
    "### Instruction:\nExplain what a paged optimizer is and why it helps memory usage.\n\n### Response:\n",
]
sample_prompts


In [ ]:
from qlora_scratch.train import run_experiment

metrics = run_experiment(PROJECT_ROOT / "data", config, sample_prompts=sample_prompts)
metrics


In [ ]:
print("Eval loss:", metrics["eval_loss"])
print("Perplexity:", metrics["perplexity"])
print("Peak VRAM (MB):", metrics["peak_vram_mb"])
print("Tokens/sec:", metrics["tokens_per_second"])


In [ ]:
for idx, (pre, post) in enumerate(zip(metrics["pre_samples"], metrics["post_samples"]), start=1):
    print(f"=== PROMPT {idx} ===")
    print(pre["prompt"])
    print("\n--- PRE-FINETUNE ---")
    print(pre["response"])
    print("\n--- POST-FINETUNE ---")
    print(post["response"])
    print("\n")
